# Multi-Paulis

In [ ]:
from collections import Counter

from IPython.display import SVG, display
from sympy import Symbol

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import FanoutCall
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit


def show_calls(graph):
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

## Multi-target CNOT

One control toggles every target. The tree accepts arbitrary initial target
states and uses no extra work wires. It applies the target-only CNOT layers
in reverse order, a single CNOT from the control, then the forward layers.
For a positive target count the depth is `2 ceil(log2(n)) + 1`.

[Trading T-gates for dirty qubits in state preparation and unitary synthesis](https://arxiv.org/abs/1812.00954), Appendix B.1.

In [ ]:
from examples.algorithms.multi_pauli_catalog import FanoutCNOTExample

### Examples

In [ ]:
n = Symbol("n")
c_multi_not_symb = FanoutCNOTExample(n)

In [ ]:
c_multi_not = FanoutCNOTExample(5)

### Graphical signatures

In [ ]:
display(SVG(field_signature_svg([
    ("symbolic fanout", c_multi_not_symb.signature, {}),
    ("five targets", c_multi_not.signature, {}),
])))
print([(port.name, port.bitsize) for port in c_multi_not_symb.signature])

### One-level call graphs

In [ ]:
show_calls(c_multi_not_symb.call_graph())
display(c_multi_not.call_graph().leaf_counts)
fanout = DynamicCircuit(6).append(c_multi_not)
display(Counter(gate.name for gate in fanout.to_ir().instructions))
display(query_qec_gate_cost(FanoutCall(5)))

## Multi-control X

The operation toggles the target when every control matches its activation
value. The AND ladder uses `n - 1` clean work wires for `n >= 2`. After the
controlled operation, the reverse ladder uses measurement and feedback to
clear the work. The same predicate path handles Y and Z.

[Constructing Large Controlled Nots](https://algassert.com/circuits/2015/06/05/Constructing-Large-Controlled-Nots.html).

In [ ]:
from examples.algorithms.multi_pauli_catalog import ControlledPauliExample

### Examples

In [ ]:
ccpauli = ControlledPauliExample("x", (1, 0, 1, 0, 1))

### Graphical signature

In [ ]:
display(SVG(field_signature_svg([("five controls", ccpauli.signature, {})])))

### One-level call graph

In [ ]:
show_calls(ccpauli.call_graph())
circuit = DynamicCircuit(10).append(ccpauli)
display(Counter(gate.name for gate in circuit.to_ir().instructions))
display(ControlledPauliExample("x", n).resource_model())